# Path predicates and auth-domination detector

`teal_path_predicates.PathPredicateAnalysis` runs a forward dataflow
with intersection at BB joins and accumulates the branch / assert
outcomes that hold on **every** path to each BB. Edge predicates per
op:

- `bnz l` / `bz l` — `(value, "nonzero" | "zero")` on the matching side.
- `assert` — `(value, "nonzero")` on the only successor.
- `switch t0 t1 … tN-1` — target k carries `(key == k)`; fall-through
  carries `(key not in [0..N-1])`.
- `match t0 t1 … tN-1` — target k carries `(key == vk)` where `vk` is the
  k-th candidate popped from the stack; fall-through carries
  `(key not in {v0, …, vN-1})`.

`teal_auth_domination.AuthDominationDetector` consumes the analysis
to flag sensitive sinks (state-mutating ops by default) that aren't
dominated by any matcher-recognised guard. Default matcher: `txn
Sender == <bytes-const>`.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

os.environ.setdefault("CODEQL", "/home/argi/tools/codeql/codeql")

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "python-analysis"))

import teal_ssa
from teal_path_predicates import PathPredicateAnalysis
from teal_auth_domination import (
    AuthDominationDetector,
    DEFAULT_SINKS, DEFAULT_MATCHERS,
    AuthSink, AuthMatcher,
)

FIXTURES = HERE.parent / "tests" / "python"

## bnz + assert example

The 3-branch fixture from `tests/path_predicates/`. Each tail BB
shows the conjunction of branch + assert outcomes on every reaching
path.

In [2]:
prog = teal_ssa.SSAProgram(FIXTURES / "path_predicates" / "db")
prog.propagate_constants()
print(PathPredicateAnalysis(prog).render(file="prog.teal"))

BB L  6-L8    (none)
BB L 12-L15   (V#1@L7 == 0)
BB L 17-L18   (V#1@L14 != 0), (V#1@L7 == 0)
BB L 20-L27   (V#1@L7 != 0)
BB L 29-L30   (V#1@L26 != 0), (V#1@L7 != 0)


## switch example

Per-target equality predicates (`key == k` for the k-th target),
fall-through gets `key not in [0..N-1]`.

In [3]:
prog = teal_ssa.SSAProgram(FIXTURES / "path_predicates_switch" / "db")
prog.propagate_constants()
print(PathPredicateAnalysis(prog).render(file="prog.teal"))

BB L  5-L7    (none)
BB L  8-L8    (V#1@L6 not in [0..2])
BB L 10-L13   (V#1@L6 == 0)
BB L 15-L18   (V#1@L6 == 1)
BB L 20-L23   (V#1@L6 == 2)
BB L 25-L27   (V#1@L6 not in [0..2])
BB L 29-L30   (none)


## match example

Per-target equality with the k-th *stack-popped* candidate.
Fall-through gets `key not in {…}` listing every candidate. With
`propagate_constants` already run, candidates that resolve to
literals render as `100, 200, 300` instead of their SSA identifiers.

In [4]:
prog = teal_ssa.SSAProgram(FIXTURES / "path_predicates_match" / "db")
prog.propagate_constants()
print(PathPredicateAnalysis(prog).render(file="prog.teal"))

BB L  6-L11   (none)
BB L 13-L13   (V#1@L10 not in {100, 200, 300})
BB L 15-L18   (V#1@L10 == 100)
BB L 20-L23   (V#1@L10 == 200)
BB L 25-L28   (V#1@L10 == 300)
BB L 30-L32   (V#1@L10 not in {100, 200, 300})
BB L 34-L35   (none)


## Auth-domination detector

Pluggable: `AuthSink` picks out which assignments need guards;
`AuthMatcher` recognises a guard pattern. Built-ins: state-mutating
ops as sinks; `txn Sender == <bytes-const>` as the matcher.

In [5]:
for case in ("vuln", "safe"):
    print(f"### {case}")
    p = teal_ssa.SSAProgram(FIXTURES / "auth_domination" / case / "db")
    p.propagate_constants()
    vs = AuthDominationDetector(p).detect()
    print(f"  {len(vs)} violation(s)")
    for v in vs:
        print(f"    {v.pretty()}")
    print()

### vuln


  1 violation(s)
    app_global_put@prog.teal:6  (state-mutating op)  preds: <no guard>

### safe


  0 violation(s)



## Adding a custom matcher: `gtxn[i] Sender == <const>`

Many real contracts authenticate against a group transaction's sender
rather than `txn Sender`. The pattern is identical except the
producing op is `gtxn`/`gtxns` instead of `txn`. Adding a new
matcher is appending to the matcher list — no detector internals
need to change.

In [6]:
from teal_path_predicates import BranchCondition
from teal_ssa import SSAVar

def _is_gtxn_sender(op) -> bool:
    if not isinstance(op, SSAVar) or op.defined_by is None:
        return False
    src = op.defined_by
    # gtxn N Sender / gtxns Sender / gtxnsa Sender …
    if src.op not in ("gtxn", "gtxns", "gtxnsa", "gtxnas"):
        return False
    toks = src.immediates.split()
    return "Sender" in toks

def _is_addr_const(op) -> bool:
    from teal_ssa import Const
    if isinstance(op, Const):
        return op.kind == "bytes"
    cv = getattr(op, "const_value", None)
    return cv is not None and cv.kind == "bytes"

def _matches_gtxn_sender_eq_const(cond: BranchCondition, prog) -> bool:
    if cond.kind != "nonzero":
        return False
    v = cond.value
    if not isinstance(v, SSAVar) or v.defined_by is None:
        return False
    a = v.defined_by
    if a.op != "==" or len(a.inputs) != 2:
        return False
    a0, a1 = a.inputs
    return ((_is_gtxn_sender(a0) and _is_addr_const(a1))
            or (_is_gtxn_sender(a1) and _is_addr_const(a0)))

GTXN_SENDER_MATCHER = AuthMatcher(
    name="gtxn[i] Sender == <const>",
    matches=_matches_gtxn_sender_eq_const,
)

# Try with the *combined* matcher set on a fixture that uses txn Sender —
# should still pass (the original matcher fires).
p = teal_ssa.SSAProgram(FIXTURES / "auth_domination" / "safe" / "db")
p.propagate_constants()
det = AuthDominationDetector(
    p, matchers=[*DEFAULT_MATCHERS, GTXN_SENDER_MATCHER],
)
vs = det.detect()
print(f"safe with combined matchers: {len(vs)} violation(s) (expected 0)")

safe with combined matchers: 0 violation(s) (expected 0)
